# 14. GCN, GraphSAGE, GAT nhẹ trên NF-BoT-IoT-v3 (task 5.2, 5.3, GĐ E)

Đọc đồ thị từ output của kernel `gnn-iot-ids-11-build-graphs-bot` (`processed/nf_bot_iot_v3/`, 13.937 đồ thị, trung bình 5,1 node mỗi cửa sổ). Mã lớp: 0 = Normal, 1 = DoS_DDoS, 2 = Truy_cap_trai_phep (không có Botnet_Backdoor, Khac). Mỗi mô hình chạy 3 seed 42, 43, 44, chọn epoch theo F1-macro trên **val** (chỉ các lớp có mặt); chỉ ghi `split=val`.

**Chạy trên CPU** (không bật GPU) vì đồ thị rất nhỏ; giữ giờ GPU cho GĐ F. Bật Internet, gắn output của notebook 11.

Các bước: (1) clone, cài gói, (2) pytest, (3) `scripts/run_gnn.py`, (4) kiểm tra tiêu chí và ghi `run_info.json`.
Đầu ra: `results_gnn_nf_bot_iot_v3.csv`, `curves_nf_bot_iot_v3.csv` (loss train, loss val, F1-macro val theo epoch), `run_info.json`; `ckpt/` ở lại trên Kaggle cho GĐ F.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"])
assert INFO["device"] == "CPU", "Notebook này dự định chạy CPU; GPU chỉ bật sau khi Phúc đồng ý"

## 1. pytest (dừng nếu có test lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Huấn luyện GCN, GraphSAGE, GAT (3 seed mỗi mô hình)

In [ ]:
DATASET = "nf_bot_iot_v3"
DATA_DIR = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{DATASET}/split.json", recursive=True)))
print("Dữ liệu:", DATA_DIR)
RESULTS = f"{WORK}/results_gnn_{DATASET}.csv"
CURVES = f"{WORK}/curves_{DATASET}.csv"
for p in (RESULTS, CURVES):
    assert not os.path.exists(p), f"{p} đã có từ trước, run_gnn sẽ nối thêm dòng trùng"

with step("train_gnn"):
    run([sys.executable, "-I", "scripts/run_gnn.py", DATA_DIR, DATASET, RESULTS, f"{WORK}/ckpt", "models=gcn,sage,gat"],
        cwd=REPO)

## 3. Kiểm tra tiêu chí 5.2, 5.3 và run_info.json

Chỉ đếm dòng theo (mô hình, seed), không tính trung bình. Độ lệch chuẩn accuracy giữa 3 seed chỉ để đối chiếu ngưỡng 1,5 điểm % (ddof = 1).

In [ ]:
import pandas as pd

res = pd.read_csv(RESULTS)
cur = pd.read_csv(CURVES)
cnt = res.groupby(["model", "seed"]).size()
print("Số dòng theo (model, seed):\n", cnt.to_string())
print("split:", sorted(res["split"].unique()), "| task:", sorted(res["task"].unique()), "| dataset:", sorted(res["dataset"].unique()))
print(res[["model", "seed", "n_params", "size_kb", "train_time_s", "infer_ms_per_window"]].to_string(index=False))

models, seeds = ["gcn", "sage", "gat"], [42, 43, 44]
curve_epochs = cur.groupby(["model", "seed"]).size()
std_acc = {m: round(float(res[res["model"] == m]["accuracy"].std(ddof=1)) * 100, 3) for m in models}
gat = res[res["model"] == "gat"]
INFO["rows_per_model_seed"] = {f"{m}_s{s}": int(cnt.get((m, s), 0)) for m in models for s in seeds}
INFO["epochs_per_model_seed"] = {f"{m}_s{s}": int(curve_epochs.get((m, s), 0)) for m in models for s in seeds}
INFO["n_params"] = {m: int(res[res["model"] == m]["n_params"].iloc[0]) for m in models}
INFO["gat_n_params"] = INFO["n_params"]["gat"]
INFO["gat_size_kb_max"] = float(gat["size_kb"].max())
INFO["std_accuracy_pp"] = std_acc
INFO["criteria"] = {
    "du_3_gnn_x_3_seed_moi_cap_1_dong": all(INFO["rows_per_model_seed"][f"{m}_s{s}"] == 1 for m in models for s in seeds) and len(res) == 9,
    "chi_split_val": set(res["split"]) == {"val"},
    "gat_params_le_100000": INFO["gat_n_params"] <= 100_000,
    "gat_file_le_1MB": INFO["gat_size_kb_max"] <= 1024,
    "co_duong_hoc_moi_lan_chay": all(INFO["epochs_per_model_seed"][f"{m}_s{s}"] >= 1 for m in models for s in seeds),
    "std_accuracy_le_1.5pp": {m: v <= 1.5 for m, v in std_acc.items()},
}
INFO["ckpt_files"] = sorted(os.listdir(f"{WORK}/ckpt"))
save_info()
print("\nrun_info.json:")
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())
run(["du", "-sh", f"{WORK}/ckpt"])
if not all(v if isinstance(v, bool) else all(v.values()) for v in INFO["criteria"].values()):
    print("CHÚ Ý: có tiêu chí không đạt, xem mục `criteria` ở trên.")